# Phase 6 — Pipeline Integration, Raw Features & Synthetic Signals 

**Phase 6.1.** Raw Features Tests (R-01 - R-07)      
**Phase 6.2.** Surprise/Uncertainty Tests (synthetic clips)(P-01 - P-06, P-10, P-11)  
**Phase 6.3.** Surprise/Uncertainty Tests (real film output required) (P07 - P09)

In [1]:
import sys, os, subprocess
import numpy as np
import pandas as pd
import cv2
import torch
from pathlib import Path
from scipy.stats import spearmanr, mannwhitneyu
os.environ.pop('SSL_CERT_FILE', None)
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
sys.path.insert(0, '/home/tamires/projects/rpp-aevans-ab/tamires/DecomposingMovies')

import cinematic_surprise.config as cfg

print(f"Python:   {sys.version}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")

FIXTURES_DIR = Path("tests/fixtures")
GOLDEN_DIR = FIXTURES_DIR / "golden"
GOLDEN_DIR.mkdir(parents=True, exist_ok=True)

print("\nSetup complete.")

I0000 00:00:1774403333.636689 1646532 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Python:   3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
PyTorch:  2.11.0+cu126
CUDA:     True

Setup complete.


In [2]:
# ── Synthetic video generator ──
import soundfile as sf

def make_synthetic_video(path, fps=24, duration_s=30, audio_sr=22050, scenario="visual"):
    W, H = 320, 240
    n_frames = fps * duration_s
    n_audio_samples = audio_sr * duration_s
    frames = []
    rng = np.random.RandomState(42)
    scroll_texture = rng.randint(0, 256, (H, W * 2, 3), dtype=np.uint8)

    for i in range(n_frames):
        t = i / fps
        if scenario == "visual":
            if t < 10:
                frame = np.full((H, W, 3), 128, dtype=np.uint8)
            elif t < 20:
                frame = np.zeros((H, W, 3), dtype=np.uint8)
                frame[:, :, 2] = 200
            else:
                frame = np.full((H, W, 3), 50 if i % 2 == 0 else 200, dtype=np.uint8)
        elif scenario == "audio":
            frame = np.full((H, W, 3), 128, dtype=np.uint8)
        elif scenario == "sync":
            frame = np.full((H, W, 3), 80, dtype=np.uint8)
            if abs(t - 5.0) < 0.1:
                frame[:, :, :] = 255
        elif scenario == "motion":
            if t < 10:
                offset = int((t / 10.0) * W)
                frame = scroll_texture[:, offset:offset+W, :].copy()
            else:
                frame = scroll_texture[:, 0:W, :].copy()
        else:
            frame = np.full((H, W, 3), 128, dtype=np.uint8)
        frames.append(frame)

    t_audio = np.linspace(0, duration_s, n_audio_samples, dtype=np.float32)
    audio = np.zeros(n_audio_samples, dtype=np.float32)
    if scenario == "audio":
        s1 = int(10 * audio_sr)
        s2 = int(20 * audio_sr)
        audio[:s1] = 0.5 * np.sin(2 * np.pi * 440 * t_audio[:s1])
        audio[s2:] = rng.randn(n_audio_samples - s2).astype(np.float32) * 0.3
    elif scenario == "sync":
        click_sample = int(5.0 * audio_sr)
        audio[click_sample:click_sample+100] = 0.9

    tmp_v = path + ".tmp_v.mp4"
    tmp_a = path + ".tmp_a.wav"
    cmd_v = ["ffmpeg","-y","-f","rawvideo","-vcodec","rawvideo","-s",f"{W}x{H}",
             "-pix_fmt","bgr24","-r",str(fps),"-i","-","-c:v","libx264",
             "-preset","ultrafast","-pix_fmt","yuv420p",tmp_v]
    proc = subprocess.Popen(cmd_v, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)
    for f in frames:
        proc.stdin.write(f.tobytes())
    proc.stdin.close()
    proc.wait()
    sf.write(tmp_a, audio, audio_sr)
    subprocess.run(["ffmpeg","-y","-i",tmp_v,"-i",tmp_a,"-c:v","copy","-c:a","aac","-shortest",path],
                   stderr=subprocess.DEVNULL)
    os.unlink(tmp_v)
    os.unlink(tmp_a)
    return path

print("Video generator ready.")

Video generator ready.


---
# Phase 6.1. Raw Features Tests (R-01 - R-07)

## R-01: Shape and Total Dimension

In [3]:
from cinematic_surprise import CinematicSurprisePipeline

vid_path = "/tmp/synth_df2.mp4"
make_synthetic_video(vid_path, scenario="visual", duration_s=30)
pipe = CinematicSurprisePipeline(max_seconds=30)
df1, df2 = pipe.run(vid_path)
print(f"df1: {df1.shape}, df2: {df2.shape}")

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 50451.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_df2.mp4':   0%|                                                              | 0/30 [00:00<?, ?s/s]W0000 00:00:1774403357.450812 1646532 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1774403357.452289 1646532 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7322 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3 MIG 1g.10gb, pci bus id: 0000:9b:00.0, compute capability: 9.0a
I0000 00:

df1: (30, 48), df2: (30, 4900)


In [4]:
dim_ok = df2.shape[1] == cfg.FEATURE_TOTAL_DIM
sync_ok = df2.shape[0] == df1.shape[0]
print(f"Columns: {df2.shape[1]} (expected {cfg.FEATURE_TOTAL_DIM})  {'✓' if dim_ok else '✗'}")
print(f"Rows sync with df1: {sync_ok}  {'✓' if sync_ok else '✗'}")
print()
if dim_ok and sync_ok:
    print("R-01: PASSED ✓")
else:
    print("R-01: FAILED ✗")

Columns: 4900 (expected 4900)  ✓
Rows sync with df1: True  ✓

R-01: PASSED ✓


## R-02: Column Naming and Channel Order

In [5]:
expected_names = cfg.feature_column_names()
actual_names = list(df2.columns)
names_match = actual_names == expected_names

print(f"Names match: {names_match}")
if not names_match:
    for i, (a, e) in enumerate(zip(actual_names, expected_names)):
        if a != e:
            print(f"  First mismatch at {i}: '{a}' vs '{e}'")
            break

offset = 0
slices_ok = True
for ch in cfg.FEATURE_CHANNEL_ORDER:
    d = cfg.FEATURE_DIMS[ch]
    first = actual_names[offset]
    last = actual_names[offset + d - 1]
    ok = first == f"{ch}_0000" and last == f"{ch}_{d-1:04d}"
    if not ok: slices_ok = False
    print(f"  {ch:<15} [{offset}:{offset+d}] dim={d:<5} {'✓' if ok else '✗'}")
    offset += d

print()
if names_match and slices_ok:
    print("R-02: PASSED ✓")
else:
    print("R-02: FAILED ✗")

Names match: True
  L1              [0:256] dim=256   ✓
  L2              [256:768] dim=512   ✓
  L3              [768:1792] dim=1024  ✓
  L4              [1792:3840] dim=2048  ✓
  semantic        [3840:4352] dim=512   ✓
  emotion         [4352:4359] dim=7     ✓
  faces           [4359:4362] dim=3     ✓
  motion          [4362:4381] dim=19    ✓
  audio_mel       [4381:4509] dim=128   ✓
  audio_spec      [4509:4515] dim=6     ✓
  audio_onset     [4515:4516] dim=1     ✓
  narrative       [4516:4900] dim=384   ✓

R-02: PASSED ✓


## R-03: No NaN or Inf in df2

In [6]:
nan_ct = df2.isna().sum().sum()
inf_ct = np.isinf(df2.values).sum()
print(f"NaN: {nan_ct}, Inf: {inf_ct}")
if nan_ct == 0 and inf_ct == 0:
    print("R-03: PASSED ✓")
else:
    offset = 0
    for ch in cfg.FEATURE_CHANNEL_ORDER:
        d = cfg.FEATURE_DIMS[ch]
        chunk = df2.iloc[:, offset:offset+d]
        n = chunk.isna().sum().sum() + np.isinf(chunk.values).sum()
        if n > 0: print(f"  {ch}: {n}")
        offset += d
    print("R-03: FAILED ✗")

NaN: 0, Inf: 0
R-03: PASSED ✓


## R-04: Static Scene → Low Feature Variance

In [7]:
all_ok = True
offset = 0
for ch in cfg.FEATURE_CHANNEL_ORDER:
    d = cfg.FEATURE_DIMS[ch]
    if ch in ['L1', 'L2', 'L3', 'L4', 'semantic']:
        grey = df2.iloc[2:9, offset:offset+d].values
        feat_std = grey.std(axis=0)
        feat_mean = np.abs(grey.mean(axis=0))
        mask = feat_mean > 1e-6
        cv = feat_std[mask] / feat_mean[mask] if mask.any() else np.array([0])
        mean_cv = cv.mean()
        ok = mean_cv < 0.05
        print(f"  {ch:<15} CV={mean_cv:.4f}  {'✓' if ok else '✗'}")
        if not ok: all_ok = False
    offset += d
print()
if all_ok:
    print("R-04: PASSED ✓")
else:
    print("R-04: FAILED ✗")

  L1              CV=0.0000  ✓
  L2              CV=0.0000  ✓
  L3              CV=0.0000  ✓
  L4              CV=0.0000  ✓
  semantic        CV=0.0000  ✓

R-04: PASSED ✓


## R-05: Grey vs Red → Different Features

In [8]:
all_ok = True
offset = 0
for ch in cfg.FEATURE_CHANNEL_ORDER:
    d = cfg.FEATURE_DIMS[ch]
    if ch in ['L1', 'L2', 'L3', 'L4']:
        grey_mean = df2.iloc[2:9, offset:offset+d].values.mean(axis=0)
        red_mean = df2.iloc[12:19, offset:offset+d].values.mean(axis=0)
        dist = np.linalg.norm(grey_mean - red_mean)
        cos = np.dot(grey_mean, red_mean) / (np.linalg.norm(grey_mean) * np.linalg.norm(red_mean) + 1e-12)
        ok = cos < 0.99
        print(f"  {ch:<6} L2={dist:.4f}  cos={cos:.4f}  {'✓' if ok else '✗'}")
        if not ok: all_ok = False
    offset += d
print()
if all_ok:
    print("R-05: PASSED ✓")
else:
    print("R-05: FAILED ✗")

  L1     L2=3.0857  cos=0.7876  ✓
  L2     L2=2.1827  cos=0.7865  ✓
  L3     L2=1.2538  cos=0.8431  ✓
  L4     L2=8.4307  cos=0.7961  ✓

R-05: PASSED ✓


## R-06: Narrative Zero-Filled Without Transcript

In [9]:
offset = 0
for ch in cfg.FEATURE_CHANNEL_ORDER:
    d = cfg.FEATURE_DIMS[ch]
    if ch == 'narrative':
        narr = df2.iloc[:, offset:offset+d].values
        norms = np.linalg.norm(narr, axis=1)
        all_zero = np.allclose(narr, 0.0)
        print(f"Narrative norms: min={norms.min():.6f} max={norms.max():.6f}")
        print(f"All zero (no transcript)?  {all_zero}  {'✓' if all_zero else '✗'}")
        if all_zero:
            print("R-06: PASSED ✓")
        else:
            print("R-06: FAILED ✗")
    offset += d

Narrative norms: min=0.000000 max=0.000000
All zero (no transcript)?  True  ✓
R-06: PASSED ✓


## R-07: Cross-Output Consistency (df1 surprise vs df2 feature delta)

In [10]:
all_ok = True
offset = 0
for ch in cfg.FEATURE_CHANNEL_ORDER:
    d = cfg.FEATURE_DIMS[ch]
    s_col = f'surprise_{ch}'
    if s_col not in df1.columns or df1[s_col].isna().all():
        offset += d
        continue
    feats = df2.iloc[:, offset:offset+d].values
    deltas = np.linalg.norm(np.diff(feats, axis=0), axis=1)
    surprise = df1[s_col].values[1:]
    valid = ~(np.isnan(surprise) | np.isnan(deltas))
    if valid.sum() > 10 and deltas[valid].std() > 1e-9 and surprise[valid].std() > 1e-9:
        r, p = spearmanr(surprise[valid], deltas[valid])
        ok = r > 0.3
        print(f"  {ch:<15} r={r:+.3f} p={p:.2e}  {'✓' if ok else '✗'}")
        if not ok: all_ok = False
    else:
        print(f"  {ch:<15} skipped (zero variance on synthetic)")
    offset += d

print()
print("Note: faces, audio channels show zero variance on synthetic (no faces, silent audio).")
print("These will be tested on real film output in Phase 7.")


  L1              r=+0.438 p=1.74e-02  ✓
  L2              r=+0.438 p=1.74e-02  ✓
  L3              r=+0.438 p=1.74e-02  ✓
  L4              r=+0.438 p=1.74e-02  ✓
  semantic        r=+0.424 p=2.20e-02  ✓
  emotion         r=+0.298 p=1.16e-01  ✗
  faces           skipped (zero variance on synthetic)
  motion          r=+0.325 p=8.52e-02  ✓
  audio_mel       skipped (zero variance on synthetic)
  audio_spec      skipped (zero variance on synthetic)
  audio_onset     skipped (zero variance on synthetic)

Note: faces, audio channels show zero variance on synthetic (no faces, silent audio).
These will be tested on real film output in Phase 7.


---
# Phase 6.2. Surprise/Uncertainty Tests (synthetic clips) (P-01 - P-06, P-10, P-11)

## P-01: Synthetic Visual — Grey/Cut/Red/Oscillation

In [11]:
from cinematic_surprise import CinematicSurprisePipeline

vid_path = "/tmp/synth_visual.mp4"
make_synthetic_video(vid_path, scenario="visual", duration_s=30)
pipe = CinematicSurprisePipeline(max_seconds=30)
df1, df2 = pipe.run(vid_path)
print(f"df1: {df1.shape}, df2: {df2.shape}")

s_grey = df1.loc[2:8, 'surprise_L1'].mean()
s_cut = df1.loc[9:11, 'surprise_L1'].max()
s_red = df1.loc[13:18, 'surprise_L1'].mean()
u_grey = df1.loc[2:8, 'uncertainty_L1'].mean()
u_osc = df1.loc[22:28, 'uncertainty_L1'].mean()

cut_spike = s_cut > s_grey * 2
u_osc_higher = u_osc > u_grey

print(f"S_L1 grey: {s_grey:.4f}, cut: {s_cut:.4f}, ratio: {s_cut/s_grey:.1f}x")
print(f"U_L1 grey: {u_grey:.4f}, oscillation: {u_osc:.4f}")
print(f"Cut spike > 2x grey?       {cut_spike}  {'✓' if cut_spike else '✗'}")
print(f"Oscillation U > grey U?    {u_osc_higher}  {'✓' if u_osc_higher else '✗'}")
print()
if cut_spike and u_osc_higher:
    print("P-01: PASSED ✓")
else:
    print("P-01: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 50533.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_visual.mp4': 100%|██████████████████████████████████████████████████| 30/30 [01:25<00:00,  2.86s/s]

df1: (30, 48), df2: (30, 4900)
S_L1 grey: 0.0012, cut: 70.4170, ratio: 59006.7x
U_L1 grey: -11.4769, oscillation: -7.8468
Cut spike > 2x grey?       True  ✓
Oscillation U > grey U?    True  ✓

P-01: PASSED ✓


## P-02: Synthetic Audio — Sine/Silence/Noise

In [ ]:
vid_path = "/tmp/synth_audio.mp4"
make_synthetic_video(vid_path, scenario="audio", duration_s=30)
pipe = CinematicSurprisePipeline(max_seconds=30)
df1, df2 = pipe.run(vid_path)

s_sine = df1.loc[3:8, 'surprise_audio_mel'].mean()
s_t1 = df1.loc[9:11, 'surprise_audio_mel'].max()
s_sil = df1.loc[13:18, 'surprise_audio_mel'].mean()
s_t2 = df1.loc[19:21, 'surprise_audio_mel'].max()

t1_spike = s_t1 > s_sine * 1.5
t2_spike = s_t2 > s_sil * 1.5

print(f"Sine: {s_sine:.4f}, Trans1: {s_t1:.4f}, Silence: {s_sil:.4f}, Trans2: {s_t2:.4f}")
print(f"Transition 1 spike?  {t1_spike}  {'✓' if t1_spike else '✗'}")
print(f"Transition 2 spike?  {t2_spike}  {'✓' if t2_spike else '✗'}")
print()
if t1_spike and t2_spike:
    print("P-02: PASSED ✓")
else:
    print("P-02: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 49165.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_audio.mp4':  80%|████████████████████████████████████████▊          | 24/30 [01:08<00:17,  2.85s/s]

## P-03: AV Sync — Flash + Click at t=5s

In [23]:
vid_path = "/tmp/synth_sync.mp4"
make_synthetic_video(vid_path, scenario="sync", duration_s=15)
pipe = CinematicSurprisePipeline(max_seconds=15)
df1, df2 = pipe.run(vid_path)

v_peak = df1['surprise_L1'].idxmax()
a_peak = df1['surprise_audio_onset'].idxmax()

v_ok = abs(v_peak - 5) <= 1
a_ok = abs(a_peak - 5) <= 1
aligned = abs(v_peak - a_peak) <= 1

print(f"Visual peak: t={v_peak}s, Audio peak: t={a_peak}s, Offset: {abs(v_peak-a_peak)}s")
print(f"Visual at 5±1s?  {v_ok}  {'✓' if v_ok else '✗'}")
print(f"Audio at 5±1s?   {a_ok}  {'✓' if a_ok else '✗'}")
print(f"Aligned ±1s?     {aligned}  {'✓' if aligned else '✗'}")
print()
if v_ok and a_ok and aligned:
    print("P-03: PASSED ✓")
else:
    print("P-03: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 51271.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_sync.mp4': 100%|████████████████████████████████████████████████████| 15/15 [00:29<00:00,  1.95s/s]

Visual peak: t=4s, Audio peak: t=4s, Offset: 0s
Visual at 5±1s?  True  ✓
Audio at 5±1s?   True  ✓
Aligned ±1s?     True  ✓

P-03: PASSED ✓


## P-04: Synthetic Motion — Scroll Then Stop

In [24]:
vid_path = "/tmp/synth_motion.mp4"
make_synthetic_video(vid_path, scenario="motion", duration_s=20)
pipe = CinematicSurprisePipeline(max_seconds=20)
df1, df2 = pipe.run(vid_path)

s_scroll = df1.loc[3:8, 'surprise_motion'].mean()
s_stop = df1.loc[9:12, 'surprise_motion'].max()
s_static = df1.loc[14:18, 'surprise_motion'].mean()

stop_spike = s_stop > s_scroll * 1.5
static_low = s_static < s_stop

print(f"Scroll: {s_scroll:.4f}, Stop: {s_stop:.4f}, Static: {s_static:.4f}")
print(f"Stop spike > 1.5x scroll?  {stop_spike}  {'✓' if stop_spike else '✗'}")
print(f"Static < stop?             {static_low}  {'✓' if static_low else '✗'}")
print()
if stop_spike and static_low:
    print("P-04: PASSED ✓")
else:
    print("P-04: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 51552.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_motion.mp4': 100%|██████████████████████████████████████████████████| 20/20 [00:39<00:00,  1.99s/s]

Scroll: 0.0020, Stop: 129.5126, Static: 0.0007
Stop spike > 1.5x scroll?  True  ✓
Static < stop?             True  ✓

P-04: PASSED ✓


## P-05: No NaN or Inf (excluding narrative without transcript)

In [25]:
vid_path = "/tmp/synth_nan.mp4"
make_synthetic_video(vid_path, scenario="visual", duration_s=15)
pipe = CinematicSurprisePipeline(max_seconds=15)
df1, df2 = pipe.run(vid_path)

float_cols = df1.select_dtypes(include=[np.floating]).columns
narr_cols = [c for c in float_cols if 'narrative' in c]
check_cols = [c for c in float_cols if c not in narr_cols]

nan_ct = df1[check_cols].isna().sum().sum()
inf_ct = np.isinf(df1[check_cols].values).sum()
narr_nan = df1[narr_cols].isna().all().all() if narr_cols else True

print(f"Checked: {len(check_cols)} cols, NaN: {nan_ct}, Inf: {inf_ct}")
print(f"Narrative all NaN (expected): {narr_nan}  {'✓' if narr_nan else '✗'}")
print()
if nan_ct == 0 and inf_ct == 0 and narr_nan:
    print("P-05: PASSED ✓")
else:
    print("P-05: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 51362.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_nan.mp4': 100%|█████████████████████████████████████████████████████| 15/15 [00:29<00:00,  1.97s/s]

Checked: 40 cols, NaN: 0, Inf: 0
Narrative all NaN (expected): True  ✓

P-05: PASSED ✓


## P-06: Surprise ≥ 0 (excluding narrative and combined)

In [26]:
vid_path = "/tmp/synth_pos.mp4"
make_synthetic_video(vid_path, scenario="visual", duration_s=15)
pipe = CinematicSurprisePipeline(max_seconds=15)
df1, df2 = pipe.run(vid_path)

s_cols = [c for c in df1.columns if c.startswith('surprise_')
          and 'narrative' not in c and 'combined' not in c]

all_ok = True
for col in s_cols:
    mn = df1[col].min()
    ok = mn >= -1e-9
    print(f"  {col:<25} min={mn:.6f}  {'✓' if ok else '✗'}")
    if not ok: all_ok = False
print()
if all_ok:
    print("P-06: PASSED ✓")
else:
    print("P-06: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 51168.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_pos.mp4': 100%|█████████████████████████████████████████████████████| 15/15 [00:28<00:00,  1.92s/s]

  surprise_L1               min=0.000000  ✓
  surprise_L2               min=0.000000  ✓
  surprise_L3               min=0.000105  ✓
  surprise_L4               min=0.000277  ✓
  surprise_semantic         min=0.000192  ✓
  surprise_motion           min=-0.000000  ✓
  surprise_emotion          min=0.000000  ✓
  surprise_faces            min=0.000000  ✓
  surprise_audio_mel        min=0.000000  ✓
  surprise_audio_spec       min=0.000000  ✓
  surprise_audio_onset      min=0.000000  ✓

P-06: PASSED ✓


## P-10: Determinism — Two Runs Identical

In [27]:
vid_path = "/tmp/synth_det.mp4"
make_synthetic_video(vid_path, scenario="visual", duration_s=10)

pa = CinematicSurprisePipeline(max_seconds=10)
df1a, df2a = pa.run(vid_path)
pb = CinematicSurprisePipeline(max_seconds=10)
df1b, df2b = pb.run(vid_path)

fc = df1a.select_dtypes(include=[np.floating]).columns
ok = True
for col in fc:
    if not np.allclose(df1a[col].values, df1b[col].values, atol=1e-9, equal_nan=True):
        print(f"  MISMATCH: {col}")
        ok = False

# Also check df2
df2_match = np.allclose(df2a.values, df2b.values, atol=1e-9)
print(f"df1 match: {ok}  df2 match: {df2_match}")
print()
if ok and df2_match:
    print("P-10: PASSED ✓")
else:
    print("P-10: FAILED ✗")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 50169.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 49531.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_det.mp4': 100%|█████████████████████████████████████████████████████| 10/10 [00:19<00:00,  1.92s/s]

df1 match: True  df2 match: True

P-10: PASSED ✓


## P-11: Golden File Regression

In [28]:
golden_df1 = GOLDEN_DIR / "golden_df1.parquet"
golden_df2 = GOLDEN_DIR / "golden_df2.parquet"

vid_path = "/tmp/synth_gold.mp4"
make_synthetic_video(vid_path, scenario="visual", duration_s=10)
pipe = CinematicSurprisePipeline(max_seconds=10)
df1c, df2c = pipe.run(vid_path)

if golden_df1.exists():
    gdf1 = pd.read_parquet(golden_df1)
    gdf2 = pd.read_parquet(golden_df2)
    fc = df1c.select_dtypes(include=[np.floating]).columns
    ok1 = all(np.allclose(df1c[c].values, gdf1[c].values, atol=1e-6, equal_nan=True) for c in fc if c in gdf1.columns)
    ok2 = np.allclose(df2c.values, gdf2.values, atol=1e-6)
    print(f"df1 match: {ok1}, df2 match: {ok2}")
    if ok1 and ok2:
        print("P-11: PASSED ✓")
    else:
        print("P-11: FAILED ✗")
else:
    df1c.to_parquet(golden_df1, index=False)
    df2c.to_parquet(golden_df2, index=False)
    print(f"P-11: FIRST RUN — Saved golden files. Re-run to verify.")
os.unlink(vid_path)

Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 51083.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing 'synth_gold.mp4': 100%|████████████████████████████████████████████████████| 10/10 [00:19<00:00,  1.96s/s]

df1 match: True, df2 match: False
P-11: FAILED ✗


---
# Phase 6.3. Surprise/Uncertainty Tests (real film output required) (P07 - P09)
Require real film pipeline output. Update paths below.

In [29]:
# ── Load real film output ──
film_output = Path("/home/tamires/projects/rpp-aevans-ab/tamires/DecomposingMovies/outputs/12_years_a_slave_surprise_uncertainty.parquet")

PF_CSV = Path("/home/tamires/projects/rpp-aevans-ab/tamires/audiovisual_stimuli/12_years_a_slave_transcript.csv")

if film_output.exists():
    df_film = pd.read_parquet(film_output)
    print(f"Loaded: {film_output.name}, {len(df_film)} seconds")
else:
    print(f"Film output not found: {film_output}")

Loaded: 12_years_a_slave_surprise_uncertainty.parquet, 7717 seconds


## P-07: Scene Cuts Correlate with L1–L4 Surprise

In [30]:
if not film_output.exists():
    print("P-07: SKIPPED")
elif 'scene_cut' not in df_film.columns:
    print("P-07: SKIPPED — no scene_cut column")
else:
    cuts = df_film[df_film['scene_cut'] == True]
    noncuts = df_film[df_film['scene_cut'] == False]
    print(f"Cuts: {len(cuts)}, Non-cuts: {len(noncuts)}")
    ok = True
    for layer in ['surprise_L1','surprise_L2','surprise_L3','surprise_L4']:
        if layer in df_film.columns:
            stat, p = mannwhitneyu(cuts[layer], noncuts[layer], alternative='greater')
            sig = p < 0.05
            print(f"  {layer}: cut_med={cuts[layer].median():.4f} noncut_med={noncuts[layer].median():.4f} p={p:.2e} {'✓' if sig else '✗'}")
            if not sig: ok = False
    print()
    print(f"P-07: {'PASSED ✓' if ok else 'FAILED ✗'}")

Cuts: 651, Non-cuts: 7066
  surprise_L1: cut_med=0.2810 noncut_med=0.0046 p=0.00e+00 ✓
  surprise_L2: cut_med=0.1753 noncut_med=0.0063 p=0.00e+00 ✓
  surprise_L3: cut_med=0.0163 noncut_med=0.0015 p=0.00e+00 ✓
  surprise_L4: cut_med=0.0200 noncut_med=0.0027 p=0.00e+00 ✓

P-07: PASSED ✓


## P-08: Narrative Surprise During Silence

In [31]:
if not film_output.exists() or not PF_CSV.exists():
    print("P-08: SKIPPED")
elif 'surprise_narrative' not in df_film.columns:
    print("P-08: SKIPPED — no surprise_narrative")
else:
    words = pd.read_csv(PF_CSV)
    words['second'] = words['start_time_new'].apply(lambda x: int(np.floor(x)))
    speech_secs = set(words['second'].unique())
    silence_secs = sorted(set(range(len(df_film))) - speech_secs)

    silence_s = df_film.loc[df_film['time_s'].isin(silence_secs), 'surprise_narrative']
    speech_s = df_film.loc[df_film['time_s'].isin(speech_secs), 'surprise_narrative']

    print(f"Speech: {len(speech_secs)} sec, Silence: {len(silence_secs)} sec")
    print(f"Silence S: mean={silence_s.mean():.6f} max={silence_s.max():.6f}")
    print(f"Speech S:  mean={speech_s.mean():.6f} max={speech_s.max():.6f}")

    TOLERANCE = 0.01
    silence_ok = silence_s.max() < TOLERANCE
    speech_dom = speech_s.mean() > silence_s.mean() * 100 if silence_s.mean() > 1e-12 else speech_s.mean() > 0.001

    print(f"Silence max < {TOLERANCE}?    {silence_ok}  {'✓' if silence_ok else '✗'}")
    print(f"Speech >> silence?            {speech_dom}  {'✓' if speech_dom else '✗'}")
    print()
    if silence_ok and speech_dom:
        print("P-08: PASSED ✓")
    else:
        print("P-08: FAILED ✗")

Speech: 2705 sec, Silence: 5012 sec
Silence S: mean=0.000069 max=0.002199
Speech S:  mean=0.000177 max=0.004014
Silence max < 0.01?    True  ✓
Speech >> silence?            False  ✗

P-08: FAILED ✗


## P-09: Interaction Adds Unique Variance

In [32]:
if not film_output.exists():
    print("P-09: SKIPPED — No real film output")
else:
    df_film = pd.read_parquet(film_output)

    channels = ['L1', 'L2', 'L3', 'L4', 'semantic', 'emotion', 'faces',
                'motion', 'audio_mel', 'audio_spec', 'audio_onset', 'narrative']

    redundant = []
    all_checked = 0

    for ch in channels:
        s_col = f'surprise_{ch}'
        u_col = f'uncertainty_{ch}'
        i_col = f'interaction_{ch}'

        if all(c in df_film.columns for c in [s_col, u_col, i_col]):
            # Skip if all NaN
            if df_film[i_col].isna().all():
                continue

            r_s = df_film[i_col].corr(df_film[s_col])
            r_u = df_film[i_col].corr(df_film[u_col])
            ok = abs(r_s) < 0.9 and abs(r_u) < 0.9
            status = '✓' if ok else '⚠ REDUNDANT'
            print(f"  {ch:<15} r(I,S)={r_s:+.3f}  r(I,U)={r_u:+.3f}  {status}")
            all_checked += 1
            if not ok:
                redundant.append(ch)

    print()
    print(f"Channels checked: {all_checked}")
    print(f"Independent:      {all_checked - len(redundant)}")
    print(f"Redundant:        {len(redundant)} — {redundant}")
    print()

    # This is a FINDING, not a code bug.
    # Some channels have near-constant uncertainty, making interaction ≈ c × surprise.
    # The library computes z(S)×z(U) correctly (confirmed in E-11).
    # Report which channels are redundant for the methods section.

    if len(redundant) == 0:
        print("P-09: PASSED ✓ — All interaction terms add unique variance")
    elif len(redundant) <= 4:
        print("P-09: PASSED WITH FINDINGS ✓")
        print(f"  {len(redundant)} channels have near-redundant interaction terms.")
        print("  This means uncertainty is nearly constant for these channels,")
        print("  so interaction ≈ constant × surprise. Not a code bug —")
        print("  document in methods section as an empirical observation.")
        print(f"  Affected: {redundant}")
    else:
        print("P-09: FAILED ✗ — Too many redundant channels (>4)")
        print("  This suggests a systematic issue with the uncertainty computation.")

  L1              r(I,S)=+0.698  r(I,U)=+0.081  ✓
  L2              r(I,S)=+0.697  r(I,U)=+0.058  ✓
  L3              r(I,S)=+0.240  r(I,U)=-0.065  ✓
  L4              r(I,S)=-0.319  r(I,U)=+0.062  ✓
  semantic        r(I,S)=-0.666  r(I,U)=-0.037  ✓
  emotion         r(I,S)=-0.582  r(I,U)=-0.041  ✓
  faces           r(I,S)=-0.994  r(I,U)=+0.036  ⚠ REDUNDANT
  motion          r(I,S)=+0.433  r(I,U)=-0.049  ✓
  audio_mel       r(I,S)=-0.995  r(I,U)=+0.058  ⚠ REDUNDANT
  audio_spec      r(I,S)=-0.763  r(I,U)=-0.083  ✓
  audio_onset     r(I,S)=-0.991  r(I,U)=+0.046  ⚠ REDUNDANT
  narrative       r(I,S)=-0.730  r(I,U)=-0.436  ✓

Channels checked: 12
Independent:      9
Redundant:        3 — ['faces', 'audio_mel', 'audio_onset']

P-09: PASSED WITH FINDINGS ✓
  3 channels have near-redundant interaction terms.
  This means uncertainty is nearly constant for these channels,
  so interaction ≈ constant × surprise. Not a code bug —
  document in methods section as an empirical observation.
  Affe